<a href="https://colab.research.google.com/github/cleophasmashiri/ai-jupter-notebooks/blob/main/gpt_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

### Roadmap

We'll build a GPT from the ground up, in the same order you'd derive it if you were inventing it yourself:

1. **Data → tokens**: turn raw text into integers a neural net can consume.
2. **A trivial baseline**: a bigram model (predict the next character from just the current one) to get the training loop and loss working end-to-end.
3. **The attention trick**: derive self-attention starting from "average the past tokens" and building up to full scaled dot-product attention.
4. **Assemble the Transformer**: stack multi-head attention + feed-forward blocks with residual connections and LayerNorm — this is the actual GPT architecture (minus scale).
5. **Train and sample**: train the full model on Shakespeare and generate new text from it.

Everything here is character-level (not word/subword) to keep the vocabulary tiny and the code simple — the ideas transfer directly to a real GPT-2/3-style BPE tokenizer.

### Glossary (reference while reading)

| Term | Meaning |
|---|---|
| **Token** | An integer id standing in for a chunk of text (here: one character). Everything the model does operates on tokens, never raw text. |
| **Vocabulary / `vocab_size`** | The number of distinct tokens the model can emit — the width of its final output layer. |
| **Embedding** | A learned lookup table mapping a token id to a dense vector (`nn.Embedding`). Turns a discrete id into something a neural net can do arithmetic on. |
| **Logits** | Raw, unnormalized scores output by the model for each possible next token — turned into probabilities by `softmax`. |
| **`block_size` / context length** | The maximum number of past tokens the model can condition on when predicting the next one. |
| **Autoregressive** | Generating a sequence one token at a time, each new token conditioned on all previous ones. |
| **Causal mask** | The mechanism (here, `tril` + `-inf` fill) that prevents a position from attending to future positions — required so training never "cheats" by seeing the answer. |
| **Q, K, V (Query, Key, Value)** | Three learned projections of a token's embedding used by attention: Q = "what I'm looking for," K = "what I offer to be matched against," V = "what I actually send if attended to." |
| **Attention head** | One instance of the Q/K/V mechanism, producing one particular way of mixing information across tokens. |
| **Multi-head attention** | Several attention heads run in parallel (on the same input) and concatenated, so the model can mix context in several different ways at once. |
| **Residual / skip connection** | Adding a layer's output back onto its input (`x = x + f(x)`) instead of replacing it — keeps gradients flowing through deep stacks. |
| **LayerNorm** | Normalizes each token's feature vector to zero mean / unit variance before a sublayer, stabilizing training. |
| **Transformer block** | One layer of self-attention + feed-forward (each wrapped in residual + LayerNorm) — GPT is a stack of these. |
| **Cross-entropy loss** | The training objective: negative log-probability the model assigned to the actual next token. Lower is better; `-ln(1/vocab_size)` is the "random guessing" baseline. |
| **`n_embd`, `n_head`, `n_layer`** | Core size hyperparameters: embedding width, number of attention heads per block, and number of stacked blocks. |

Refer back to this table any time a term feels unfamiliar further down — it isn't meant to be memorized up front.

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2023-01-17 01:39:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2023-01-17 01:39:28 (29.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:   # open the downloaded file, decoding it as UTF-8
    text = f.read()                                    # read the whole file into one big Python string

In [ ]:
print("length of dataset in characters: ", len(text))   # len() on a string counts characters

length of dataset in characters:  1115394


In [ ]:
# let's look at the first 1000 characters
print(text[:1000])   # slice the first 1000 characters to eyeball the raw text

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))   # set(text) dedups characters; sorted() gives a fixed, reproducible ordering
vocab_size = len(chars)            # size of our "alphabet" -> this becomes the width of the model's output layer later
print(''.join(chars))              # show the actual character set
print(vocab_size)                  # 65 for tiny shakespeare


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


**Tokenization, in the simplest possible form.** A neural net can't consume raw text — it needs integers (and eventually vectors). Here we use the crudest possible tokenizer: one token per *character*. `vocab_size` (65) is the number of distinct symbols the model will ever need to predict over — it directly sets the width of the model's output layer (a probability distribution over 65 classes at every position).

Real GPT models use *subword* tokenizers (e.g. GPT-2's BPE, ~50k tokens) instead of characters — that's a tradeoff between sequence length and vocabulary size: fewer, richer tokens mean shorter sequences per unit of text (cheaper, since attention is O(T²)), at the cost of a bigger, sparser output layer and a separate tokenizer-training step. Character-level keeps this notebook self-contained.

In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }   # string-to-index: char -> its position in `chars`
itos = { i:ch for i,ch in enumerate(chars) }   # index-to-string: inverse mapping, position -> char
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))          # e.g. [46, 47, 47, 1, 58, 46, 43, 56, 43]
print(decode(encode("hii there")))  # round-trip check: encode then decode should return the original string

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


`stoi`/`itos` are just the two halves of a bijection between characters and integers in `[0, vocab_size)`. `encode`/`decode` are the tokenizer's `.encode()`/`.decode()` in miniature — this *is* what `tiktoken` or a HuggingFace tokenizer does, just without merge rules. Keep this mental model: everywhere below, "token" means "integer id from this table," and the model never sees a character — only these ids (and later, embeddings of these ids).

In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)   # encode() gives a list[int]; wrap it as a 1D int64 tensor
print(data.shape, data.dtype) # torch.Size([1115394]) torch.int64 -- one long sequence of token ids, no more "text"
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

The entire dataset becomes one long 1D tensor of integer ids, `dtype=torch.long` (i.e. int64) because `nn.Embedding` and `F.cross_entropy` both require integer *index* tensors, not floats. Nothing about the text's structure (sentences, characters, punctuation) survives here except order — the model has to (re)discover Shakespeare's structure purely from this sequence of integers.

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]   # everything before index n
val_data = data[n:]     # everything from index n onward -- a later, non-overlapping slice of the text

Note this is a **sequential** split (first 90% / last 10%), not a random shuffle. Two reasons: (1) it prevents leakage — random windows would let validation chunks overlap almost entirely with training chunks just a few characters away; (2) it mimics the real evaluation setting for a language model — "trained on the past, tested on what comes after" — rather than an i.i.d. assumption that doesn't really hold for text.

In [ ]:
block_size = 8                  # max number of tokens of context the model looks at when predicting the next one
train_data[:block_size+1]       # +1 because a chunk of 9 tokens yields block_size (input, target) pairs -- see next cell

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

`block_size` (aka **context length** / `n_ctx` in GPT papers) is the maximum number of previous tokens the model is allowed to look at when predicting the next one. It's a hyperparameter you choose, not something inherent to the data — increasing it lets the model condition on more history, at the cost of attention's O(block_size²) compute and memory. GPT-3 uses 2048, modern long-context models use tens of thousands to millions; here we use 8 (later bumped to 32) purely so the toy examples below print legibly.

In [ ]:
x = train_data[:block_size]        # inputs: tokens 0..7
y = train_data[1:block_size+1]     # targets: tokens 1..8, i.e. x shifted one position to the right
for t in range(block_size):
    context = x[:t+1]              # everything seen "so far", growing from length 1 up to block_size
    target = y[t]                  # the single next token that should follow `context`
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


This loop is the key insight behind training-example construction: a single chunk of `block_size+1` tokens yields `block_size` *separate* training examples, one per prefix length (context of length 1, 2, 3, ... up to `block_size`). This is what makes the model useful at inference time with any context length from 1 up to `block_size` — it was explicitly trained on all of those lengths, not just the full one. It's also why the target `y` is just `x` shifted by one position: next-token prediction is the entire task, and every position in a sequence supplies its own training signal for free.

# Understanding `torch.randint()`

`torch.randint()` generates random integers within a specified range. It is commonly used in PyTorch to randomly sample indices, labels, or batches of data — exactly how `get_batch` (next cell) picks where each training chunk starts.

### Syntax

```python
torch.randint(low, high, size)
# or
torch.randint(high, size)
```

**Parameters:**
- `low` (optional): lowest integer, inclusive. Defaults to 0.
- `high`: highest integer, exclusive.
- `size`: shape of the output tensor.

### Example 1 — basic usage

```python
import torch
torch.randint(10, (5,))
```
```
tensor([3, 8, 1, 5, 7])
```
Generates 5 random integers, each in `[0, 10)` — 10 itself is excluded. Think of it as randomly picking five numbers from `0 1 2 3 4 5 6 7 8 9`.

### Example 2 — specify a lower bound

```python
torch.randint(5, 10, (6,))
```
```
tensor([6, 8, 5, 9, 7, 6])
```
Values come from `5 6 7 8 9`. `5` is included, `10` is excluded — the interval is `[5, 10)`, where `[` means inclusive and `)` means exclusive.

### Example 3 — a 2D tensor

```python
torch.randint(0, 100, (3, 4))
```
```
tensor([[12, 45, 98,  2],
        [77,  5, 16, 63],
        [44, 29, 81, 70]])
```
`size=(3, 4)` produces 3 rows and 4 columns of independent random draws.

### The line from `get_batch`

```python
batch_size = 4
block_size = 8
ix = torch.randint(len(data) - block_size, (batch_size,))
```

Suppose (for illustration) `len(data) = 1000`. Then `len(data) - block_size = 1000 - 8 = 992`, so PyTorch executes:

```python
ix = torch.randint(992, (4,))   # equivalent to torch.randint(0, 992, (4,))
```
```
tensor([101, 540, 220, 905])
```
These four numbers are random **starting positions** inside the dataset. (In the real notebook, `len(data)` is `train_data`'''s actual length, ~1,003,854 — the arithmetic is identical, just with a bigger number.)

### Why subtract `block_size`?

Later in the code: `data[i : i + block_size]` — each training example needs `block_size` consecutive tokens starting at `i`. If `i = 998` (with `len(data)=1000`), then `998 + 8 = 1006`, which runs past the end of the dataset. Capping `i` at `991` gives `991 + 8 = 999` — the last valid index — so the slice always stays inside the dataset.

### Visual example

Dataset indices: `0 1 2 3 4 5 6 7 8 9 ... 999`

Suppose `ix = tensor([5, 20, 100, 300])`. These become the starting positions:

```
Example 1: 5   6   7   8   9   10  11  12
Example 2: 20  21  22  23  24  25  26  27
Example 3: 100 101 102 103 104 105 106 107
Example 4: 300 301 302 303 304 305 306 307
```
Each sequence contains exactly 8 tokens.

### Why is the output shape `(batch_size,)`?

```python
batch_size = 4
torch.randint(992, (4,))
```
```
tensor([15, 201, 700, 512])
```
Each integer is one random starting index, used here:

```python
x = torch.stack([data[i : i + block_size] for i in ix])
```
which is equivalent to stacking:
```
data[15  : 23]
data[201 : 209]
data[700 : 708]
data[512 : 520]
```
After stacking, `x` has shape `(4, 8)` — 4 sequences (batch size) of 8 tokens each (block size).

### Summary

| Call | Result |
|---|---|
| `torch.randint(10, (5,))` | 5 random integers from 0 to 9 |
| `torch.randint(5, 10, (5,))` | 5 random integers from 5 to 9 |
| `torch.randint(100, (2, 3))` | a 2×3 matrix of random integers from 0 to 99 |
| `torch.randint(len(data) - block_size, (batch_size,))` | random starting positions for training sequences that fit completely inside the dataset |

In language model training, `torch.randint()` is used to randomly sample starting positions so every training step sees a different slice of the dataset. This randomness helps the model learn more effectively and prevents it from always training on the data in the same order.

In [ ]:
torch.manual_seed(1337)          # fix the RNG so torch.randint below is reproducible across runs
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))       # batch_size random starting offsets into `data`
    x = torch.stack([data[i:i+block_size] for i in ix])             # stack batch_size chunks of length block_size -> (B, T)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])         # same chunks shifted by 1 position -> the targets, (B, T)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)      # (4, 8) = (batch_size, block_size)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]      # prefix of length t+1 from batch element b
        target = yb[b,t]           # the token that should follow it
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

### `get_batch`, line by line

This function is the actual data loader — every training step calls it once to get a fresh minibatch. Worth understanding exactly, since bugs in a from-scratch training loop tend to live here.

```python
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
```

- **`data = train_data if split == 'train' else val_data`** — pick which 1D token tensor to sample from, depending on whether the caller wants a training or validation batch. `train_data`/`val_data` are the same sequential split from earlier — no shuffling happens here either.
- **`ix = torch.randint(len(data) - block_size, (batch_size,))`** — draw `batch_size` random integers, each uniform in `[0, len(data) - block_size)`. This is *where each sampled chunk starts*. The upper bound is deliberately `len(data) - block_size`, not `len(data)`: the largest chunk we slice is `data[i : i+block_size]`, and its last target position needs `data[i+block_size]` to exist (see `y` below) — so `i` must never get closer than `block_size` tokens to the end of `data`, or the slice would silently come up short. `(batch_size,)` is the *shape* of the output — one random start per example in the batch.
- **`x = torch.stack([data[i:i+block_size] for i in ix])`** — for every sampled start `i`, slice out `block_size` consecutive tokens. Each slice is a 1D tensor of shape `(block_size,)`; the list comprehension produces `batch_size` of them; `torch.stack` stacks them along a new leading dimension, giving one 2D tensor of shape `(batch_size, block_size)` — this is `x`, i.e. `B` sequences of length `T`.
- **`y = torch.stack([data[i+1:i+block_size+1] for i in ix])`** — the crucial trick: **reuse the exact same `ix`**, but slice starting one token later. Since `x[b] = data[i : i+block_size]` and `y[b] = data[i+1 : i+block_size+1]`, `y[b]` is *literally* `x[b]` shifted left by one position — so `y[b, t]` is always the token that comes right after `x[b, t]` in the original text. That is the entire label-construction step for next-token prediction: no separate labels are ever created, they are just an offset view of the same data.
- **`return x, y`** — both shape `(batch_size, block_size)`, ready to feed straight into the model'''s `forward(x, y)`.

**Things that are easy to miss:**
- Sampling is done **with replacement, every call** — there is no concept of an "epoch" here (going through all the data once, without repeats). Each training step draws a fresh set of random windows, some of which may overlap with windows used in earlier or even the *same* step. Over `max_iters` steps this still covers the data thoroughly — this is standard for a from-scratch demo like this; production training loops often add proper epoch-based shuffling instead.
- **Every row in `y` reuses the same starting indices as `x`** (`ix`) — that is what keeps input/target pairs aligned. A common bug when modifying this code is drawing separate random indices for `x` and `y`, which silently produces garbage (uncorrelated) training pairs.
- The bound `len(data) - block_size` (not `len(data) - block_size - 1`) works because `torch.randint(high, ...)` samples from `[0, high)` — `high` itself is excluded — so the largest possible `i` is already `len(data) - block_size - 1`, exactly what is needed so `i + block_size` never runs past the last index of `data`.

The version inside the full model further down is identical except for one added line, `x, y = x.to(device)`, which moves the batch onto the GPU if one is available.

In [ ]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


Stacking many `(context, target)` pairs into a batch (`B` = batch dimension) is purely a GPU-efficiency trick — the examples are otherwise independent and never interact with each other (that stays true throughout this notebook: attention lets tokens within a sequence talk to each other, but never across batch elements). So every tensor from here on has shape `(B, T, ...)`: `B` = batch, `T` = time/sequence position (≤ `block_size`), and later `C` = channel/embedding dimension. Keep track of these three axes — nearly every bug in transformer code is a shape mismatch between them.

### Checkpoint: data & tokenization

Before moving on, make sure you can answer these without scrolling back — try each one, then click to reveal the answer.

1. Why is `vocab_size` exactly the width of the model's final output layer?

<details><summary>Show answer</summary>

At every position the model's job is to output a probability distribution over "what token comes next," and there are exactly `vocab_size` possible next tokens. The final layer (the embedding table itself in the toy bigram model, or `lm_head` in the full model) has to emit one logit per possible token — so its output width must equal `vocab_size`.

</details>

2. Why is the train/val split done sequentially instead of with a random shuffle?

<details><summary>Show answer</summary>

Two reasons: (1) it prevents leakage — random windows would let validation chunks overlap almost entirely with training chunks just a few characters away; (2) it mimics the real evaluation setting for a language model ("trained on the past, tested on what comes after") rather than assuming the data is i.i.d., which text isn't.

</details>

3. If `block_size = 8`, how many distinct `(context, target)` training examples does a single 9-token chunk produce, and why?

<details><summary>Show answer</summary>

8 examples. A chunk of `block_size + 1` tokens gives one `(context, target)` pair for every prefix length from 1 up to `block_size` — context of length 1→2, length 2→3, ..., length 8→9. That's what lets the trained model handle any context length from 1 to `block_size` at inference time, not just the full one.

</details>

4. Why does every tensor from here on have a leading batch dimension `B`, even though batch elements never interact with each other?

<details><summary>Show answer</summary>

Purely a GPU-efficiency trick. Stacking many independent `(context, target)` pairs into one tensor lets a single vectorized forward/backward pass process all of them at once, instead of looping one example at a time. The examples never interact with each other — that stays true even once attention is introduced, since attention mixes information across the `T` (time) axis but never across the `B` (batch) axis.

</details>

If any of these aren't automatic, revisit the corresponding note above before continuing — everything past this point builds on it.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)   # a (vocab_size, vocab_size) table: row i = logits for "what follows token i"

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)   -- looks up one row of the table per token id in idx

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)     # flatten batch & time together -> (N, C), since cross_entropy wants one row per example
            targets = targets.view(B*T)      # flatten targets to match -> (N,)
            loss = F.cross_entropy(logits, targets)   # -mean(log softmax(logits)[correct class]) over all N positions

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)              # forward pass on the whole sequence generated so far
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)          -- only the next-token prediction is needed
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)          -- turn logits into a valid probability distribution
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)   -- randomly draw one token id per batch row, weighted by probs
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)  -- grow the sequence by one token and repeat
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)     # (32, 65) = (batch_size*block_size, vocab_size)
print(loss)             # ~4.87, close to -ln(1/65)=4.17 since the model starts out close to random

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


**Why "bigram"?** `nn.Embedding(vocab_size, vocab_size)` is literally a `vocab_size × vocab_size` lookup table: row `i` holds the (unnormalized) logits for "what comes after token `i`". So this model predicts the next character using *only* the current character — no context at all, hence "bigram" (a 2-gram: previous token → next token). It's the simplest possible language model, useful here purely as scaffolding to get the training loop and generation loop working before adding any real context-mixing.

**Loss.** `logits` has shape `(B,T,vocab_size)` — for every position, an unnormalized score per possible next character. `F.cross_entropy` applies `softmax` then negative-log-likelihood internally:

$$\mathcal{L} = -\frac{1}{N}\sum_i \log \frac{e^{\text{logit}_{i,\,y_i}}}{\sum_j e^{\text{logit}_{i,j}}}$$

It requires a 2D `(N, C)` input and a 1D `(N,)` target, hence the `.view(B*T, C)` / `.view(B*T)` reshape — batch and time get flattened together because cross-entropy only cares about "which of the `C` classes is correct at each of the `N` positions," not which batch/time index that position came from. A uniform-random model over 65 classes gives loss `-log(1/65) ≈ 4.17` — compare that to the `4.88` printed above (untrained, in the same ballpark) and the `1.66` the full model reaches after training later on.

**`generate`.** This is autoregressive sampling — the same loop every LLM inference server runs: (1) run the model forward, (2) keep only the logits at the *last* position (`logits[:, -1, :]`), since that's the only "next token" prediction needed right now, (3) `softmax` to turn logits into a probability distribution, (4) `torch.multinomial` to sample one token from that distribution (stochastic, not `argmax` — so output varies run to run), (5) append the sampled token and repeat. Starting from `torch.zeros((1,1))` means "start generation from token id 0" (which happens to decode to `\n`).

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)   # AdamW: adaptive per-parameter learning rates + decoupled weight decay

AdamW is Adam with decoupled weight decay — the standard optimizer choice for Transformers (vanilla SGD converges far slower on this kind of loss landscape). `lr=1e-3` is fairly high for AdamW on a real model, but fine here since this bigram model has only ~4k parameters and no depth to destabilize.

In [ ]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)   # clear old .grad values so they don't accumulate onto this step's gradients
    loss.backward()                          # backprop: populate .grad on every parameter via autodiff
    optimizer.step()                         # apply one AdamW update using those gradients

print(loss.item())


4.65630578994751


The four-line training step is the same in essentially every PyTorch model you'll write: **forward** (compute `loss`), **`zero_grad`** (clear stale gradients left from the previous step — otherwise they'd silently accumulate), **`backward`** (populate `.grad` on every parameter via autodiff), **`step`** (have the optimizer nudge parameters using those gradients). `set_to_none=True` is a minor speed/memory optimization — it frees gradient tensors instead of overwriting them with zeros.

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


Still gibberish — expected. A bigram model's ceiling is low no matter how long you train it: each character is predicted from exactly one character of context, so it can learn plausible letter-*pairs* (e.g. "q" tends to be followed by "u") but nothing about words, grammar, or long-range structure. Fixing this requires the model to actually use its context window — which is exactly what attention provides, starting in the next section.

## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))           # lower-triangular 3x3 of 1s: row i has i+1 ones (cols 0..i), rest 0
a = a / torch.sum(a, 1, keepdim=True)      # normalize each row to sum to 1 -> row i is a uniform average over rows 0..i
b = torch.randint(0,10,(3,2)).float()      # some arbitrary data to average, shape (3,2)
c = a @ b                                  # (3,3) @ (3,2) -> (3,2): row i of c = mean of rows 0..i of b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


This is the mathematical core the rest of self-attention builds on, so it's worth sitting with. `a` is lower-triangular and each row sums to 1 (a *row-stochastic* matrix), so `a @ b` computes, for row `i`, a weighted average of rows `0..i` of `b` — using only *past and current* rows, never future ones. Concretely, here every past row gets equal weight (`1/(i+1)`), i.e. a running mean. The punchline we're driving toward: if instead of uniform weights we make the weights **data-dependent** (computed from the vectors themselves rather than fixed constants), this exact "triangular weighted average" mechanism *is* causal self-attention.

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)     # random data standing in for token embeddings: 4 sequences, 8 positions each, 2 features
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)                 -- every position up to and including t, for this batch element
        xbow[b,t] = torch.mean(xprev, 0)           # average those rows together -> a single (C,) vector


`xbow[b,t]` = the running mean, over time, of `x[b, 0..t]` — a "bag of words" up to the current position (hence `bow`). This is written as an explicit double loop (an O(T) mean, computed at each of T positions) purely for clarity; the next two cells show how to get the identical result with a single matrix multiply, which is both faster and — more importantly — the form that generalizes directly to attention.

In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))            # (T,T) lower-triangular mask: 1 where j<=i, 0 where j>i
wei = wei / wei.sum(1, keepdim=True)           # normalize rows to sum to 1 -> same uniform-average weights as before
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)      -- wei broadcasts identically over every batch element
torch.allclose(xbow, xbow2)                    # confirms this matmul reproduces the double-loop result exactly

True

Same result, `wei @ x` instead of nested loops: `wei` is exactly the `a` matrix from the earlier toy example (lower-triangular, row-normalized), broadcast over the batch dimension — `(T,T) @ (B,T,C) → (B,T,C)`. This is purely a vectorization win right now (`wei` is still fixed/uniform), but notice the *shape* of the computation (`weights @ values`) is already in the exact form attention needs.

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))            # same causal mask, now used as a boolean template rather than weights directly
wei = torch.zeros((T,T))                       # start from all-zero "affinities" (no data-dependence yet)
wei = wei.masked_fill(tril == 0, float('-inf'))  # future positions (j>i) get -inf so softmax will zero them out
wei = F.softmax(wei, dim=-1)                   # softmax of [0,...,0,-inf,...] over the allowed entries -> uniform weights again
xbow3 = wei @ x
torch.allclose(xbow, xbow3)                    # same result a third way -- and THIS form is what generalizes to real attention


True

Reformulating the fixed averaging matrix via `masked_fill(-inf)` + `softmax` looks like more work for the same answer, but it's the version that generalizes: `softmax` of all-zeros-except-masked gives uniform weights over the unmasked entries (same result as before), but if the *pre-softmax* values (`wei`, before masking) aren't uniform zeros and are instead computed from the data — that's the entire jump to real attention, made in the very next cell. The mask itself (`tril == 0 → -inf`) is what makes this **causal**: position `t` can only attend to positions `≤ t`, which is required for autoregressive training (the model must never get to peek at future tokens it's supposed to predict).

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)      # projects each token's C-dim vector into a head_size-dim "key"
query = nn.Linear(C, head_size, bias=False)    # ...into a head_size-dim "query"
value = nn.Linear(C, head_size, bias=False)    # ...into a head_size-dim "value" (what actually gets aggregated)
k = key(x)   # (B, T, 16)                       -- every token's key vector
q = query(x) # (B, T, 16)                       -- every token's query vector
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)   -- wei[b,i,j] = q_i . k_j: how much i "wants" to attend to j

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))    # causal mask: token i cannot see tokens j>i (future positions)
wei = F.softmax(wei, dim=-1)                       # each row -> a probability distribution over allowed positions

v = value(x)                                       # (B,T,16), the actual content each token contributes if attended to
out = wei @ v                                      # (B,T,T) @ (B,T,16) -> (B,T,16): weighted sum of values per query position
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

**This cell is the entire idea of the Transformer.** Walk through it slowly:

- `key`, `query`, `value` are three independent linear projections of the same input `x`. Each token emits a **query** ("what am I looking for?"), a **key** ("what do I contain, for others to find?"), and a **value** ("what do I actually communicate, if attended to?"). Q and K live in the same space so they can be dot-producted; V doesn't have to.
- `wei = q @ k.transpose(-2,-1)` computes, for every pair of positions `(i,j)`, the dot product `q_i · k_j` — high when query `i` and key `j` "match." This is the (unnormalized, uncausaled) **attention score matrix**, shape `(T,T)`.
- Masking + softmax turns each row into a probability distribution over *allowed* (`j ≤ i`) positions — "how much should position `i` attend to each earlier position `j`."
- `out = wei @ v`: each output position becomes a weighted average of *value* vectors, weighted by how much attention that position pays to each earlier one.

The key upgrade over the previous section: **the weights are now data-dependent** — computed from `x` itself via learned projections — rather than fixed/uniform. That's the entire conceptual leap from "running mean" to "attention." Formally, for one head:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

where $M$ is the additive causal mask ($-\infty$ above the diagonal, $0$ on/below it) and $d_k$ = `head_size`. This cell doesn't yet include the $1/\sqrt{d_k}$ scaling — that's introduced (and motivated) a few cells below. Also note: this is "self"-attention because Q, K, and V all come from the same `x` — see the note further down on cross-attention for the alternative.

In [ ]:
wei[0]  # attention weights for the first sequence in the batch: row t = weights token t assigns to tokens 0..t

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Row `t` of this matrix is literally "how much token `t` attends to each earlier token" (and itself), summing to 1. Note it's *already* non-uniform despite the model being randomly initialized and untrained — e.g. row 3 puts 58% of its weight on position 0. Once trained, these weights become interpretable-ish (a token often attends heavily to the subject of its clause, or a matching quote/bracket) — this is the basis of the attention-map visualizations you may have seen for real language models.

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)                          # unit-variance random keys (simulating well-initialized projections)
q = torch.randn(B,T,head_size)                           # unit-variance random queries
wei = q @ k.transpose(-2, -1) * head_size**-0.5          # scale by 1/sqrt(head_size) to counteract the variance blow-up from the dot product

In [ ]:
k.var()  # ~1.0, as expected for randn

tensor(1.0449)

In [ ]:
q.var()  # ~1.0, as expected for randn

tensor(1.0700)

In [ ]:
wei.var()  # ~1.0 too, now that scaling is applied (compare to ~16 without it, in the earlier cell)

tensor(1.0918)

**Why scale by `1/sqrt(head_size)`?** If `q` and `k` have unit-variance entries (a reasonable initialization target), each dot product `q_i · k_j` sums `head_size` roughly-independent unit-variance terms, so `Var(q·k) ≈ head_size` — i.e. `wei` ends up with variance ≈ `head_size` (here ≈16), not 1, as confirmed by `wei.var()` above. Dividing by `sqrt(head_size)` restores unit variance regardless of `head_size`, which matters because...

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)   # mild logits -> a fairly "soft" (spread out) distribution

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # scaling logits by 8 before softmax gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

...unscaled, large-magnitude logits make `softmax` saturate toward one-hot, as shown: multiplying by 8 turns a mild preference into an 80%-confident near-certainty. A saturated softmax has near-zero gradient almost everywhere (it's already near its extreme), which stalls learning — and gets worse the larger `head_size` is, since that's exactly what inflates the pre-scaling variance. Dividing by $\sqrt{d_k}$ keeps the softmax "diffuse" (soft) regardless of head size, which is why every Transformer implementation includes it — this full recipe is why it's called **scaled** dot-product attention.

### Checkpoint: self-attention derivation

1. Starting from the lower-triangular averaging matrix, what's the one change that turns "running mean of the past" into real attention?

<details><summary>Show answer</summary>

Making the averaging weights **data-dependent** instead of fixed/uniform. The lower-triangular matrix already gives a causal (only-look-at-the-past) weighted average — attention just replaces the fixed weights `1/(i+1)` with weights computed from the data itself, via `softmax(Q @ K^T)`.

</details>

2. Why must the causal mask be applied *before* the softmax, not after?

<details><summary>Show answer</summary>

Softmax needs to see `-inf` at the masked (future) positions so that, after exponentiating, those positions get *exactly* zero probability and every row still sums to 1. If you masked after softmax instead (e.g. zeroing out entries post-hoc), the remaining "allowed" probabilities would no longer sum to 1 — you'd need to renormalize, and you'd have thrown away the numerically clean `-inf → 0` guarantee.

</details>

3. What does each of Q, K, and V represent, and why don't K and V need to live in the same vector space as each other (only Q and K do)?

<details><summary>Show answer</summary>

Q = "what am I looking for," K = "what do I offer, to be matched against," V = "what do I actually send if attended to." Q and K must share a space because they're combined via a dot product (`Q @ K^T`) to produce a compatibility score. V never gets dot-producted with anything — it's only ever weighted-summed (`wei @ V`) — so its dimensionality is independent and can differ (in this notebook it doesn't, but it's allowed to).

</details>

4. Why does an unscaled `q @ k.T` have variance approximately equal to `head_size`, and what specifically goes wrong if you skip the `1/sqrt(head_size)` scaling?

<details><summary>Show answer</summary>

Each entry of `q @ k.T` is a dot product summing `head_size` terms; if `q` and `k` have unit-variance entries, those terms are roughly independent, so their variances add up to `≈ head_size`. Left unscaled, that inflated variance pushes softmax's input logits to large magnitudes, which makes softmax saturate toward one-hot — near-zero gradient almost everywhere, which stalls learning (and gets worse the larger `head_size` is).

</details>

Try answering purely from the derivation above (the cells building up from the triangular-matrix toy example) before checking your answers against it again.

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps                    # small constant to avoid dividing by zero when variance is ~0
    self.gamma = torch.ones(dim)      # learnable per-feature scale, initialized to 1 (starts as a no-op)
    self.beta = torch.zeros(dim)      # learnable per-feature shift, initialized to 0 (starts as a no-op)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean                 -- mean over dim 1 = over FEATURES, per example (not per batch!)
    xvar = x.var(1, keepdim=True) # batch variance                -- variance over that same axis
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance   -- each row now has mean 0, std 1
    self.out = self.gamma * xhat + self.beta          # learned affine transform: lets the network undo the normalization if useful
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

LayerNorm normalizes **each individual example's feature vector** to zero mean / unit variance (here: `x.mean(1)` — averaged over the *feature* dimension, per row), then applies a learned affine transform (`gamma`, `beta`) so the network can undo the normalization if that's actually better for a given layer. Contrast with BatchNorm, which normalizes each *feature* across the batch (`mean(0)` instead of `mean(1)`) — that makes BatchNorm's statistics depend on which other examples happen to share the batch (bad for variable/small batch sizes, and awkward for sequence models that may run batch size 1 at inference). LayerNorm has no cross-example dependency, which is why Transformers use it almost exclusively. (The comment `# used to be BatchNorm1d` is a nod to an earlier notebook in this series that used BatchNorm for MLPs — same code shape, just the normalization axis swapped.)

Formally, per row $x \in \mathbb{R}^d$:
$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2+\epsilon}}, \qquad y = \gamma \odot \hat{x} + \beta$$

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs -- NOT the axis LayerNorm normalizes, so not exactly (0,1)

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features -- this IS the axis LayerNorm normalizes, so ~(0,1)

(tensor(-9.5367e-09), tensor(1.0000))

This confirms the two normalization axes are different: the first check (`x[:,0]`, one feature across the whole batch) is *not* standardized — mean ≈0.15, std ≈0.88 — because that's not the axis LayerNorm normalizes; the second (`x[0,:]`, all features of one example) is essentially exactly standardized (mean ≈0, std =1), because that *is* the axis LayerNorm operates on.

In [ ]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



This is a placeholder illustrating **cross-attention** and the encoder/decoder split from the original "Attention Is All You Need" paper: an *encoder* stack processes the source sentence (no causal mask — every token can see every other, since the whole sentence is known upfront) into key/value vectors; a *decoder* stack generates the target sentence autoregressively (causal mask, like our `Head` above), and at each decoder layer, queries come from the decoder's own sequence while keys/values come from the *encoder's* output — "cross" because Q comes from one sequence and K/V come from another. GPT is decoder-only (no encoder, no cross-attention): every `Head` in this notebook is self-attention because Q, K, and V all derive from the same `x`. This distinction only matters if you build a translation/seq2seq model instead of a plain autoregressive LM like the one we finish below.

### Full finished code, for reference

You may want to refer directly to the git repo instead though.

### How the pieces below fit together

This assembles everything above into an actual (tiny) GPT. Reading order for the classes, bottom-up:

- **`Head`** — exactly the self-attention head derived above, now parameterized by the real hyperparameters (`n_embd`, `block_size`, `dropout`) instead of the toy numbers used earlier. `register_buffer('tril', ...)` stores the causal mask as part of the module's state (moves with `.to(device)`, saved/loaded with the model) without making it a trainable parameter.
- **`MultiHeadAttention`** — runs `num_heads` independent `Head`s in parallel on the *same* input, concatenates their outputs, then projects back to `n_embd` with a linear layer. Why multiple heads instead of one bigger head? Each head can specialize (e.g. one tracks local syntax, another tracks long-range coreference) — concatenation lets the model combine several different attention patterns per layer instead of being forced into one. Note `num_heads * head_size == n_embd`, so the concatenation exactly refills the embedding dimension.
- **`FeedFoward`** — a per-position 2-layer MLP (`n_embd → 4·n_embd → n_embd` with a ReLU in between) applied identically and independently to every token. If attention is where tokens *exchange* information, the feed-forward is where each token *processes* what it just received — no cross-token interaction happens here at all.
- **`Block`** — one Transformer layer: `x = x + SelfAttention(LayerNorm(x))`, then `x = x + FeedForward(LayerNorm(x))`. Two things matter here: (1) the **residual/skip connections** (`x + ...`) — without them, gradients through many stacked layers vanish/explode; with them, each layer only needs to learn a small *update* to `x` rather than reproducing `x` from scratch, and the original `x` keeps a clean gradient path all the way back to the input. (2) **Pre-norm** (LayerNorm applied *before* each sublayer, not after, unlike the original 2017 paper) — this is the change GPT-2 onward adopted because it makes deep Transformers noticeably more stable to train.
- **`BigramLanguageModel`** (full version) — same class name as the toy model, now with real machinery: token embeddings **and** learned positional embeddings (`position_embedding_table`), summed together (this is how the model knows position at all — recall from the attention notes above that attention itself is permutation-invariant and has "no notion of space"), a stack of `n_layer` `Block`s, a final LayerNorm, then a linear `lm_head` projecting `n_embd → vocab_size` to get logits. `forward` and `generate` are structurally identical to the bigram version — that's the whole point of this exercise: *attention slots into the exact same training/generation scaffolding*, it just makes each token's representation actually depend on its context.
- **`estimate_loss`** — averages loss over `eval_iters` batches (instead of just one) for a less noisy read on train/val loss, and toggles `model.eval()`/`model.train()` around it — this matters once `dropout` (or BatchNorm) is in play, since those layers behave differently at eval time.

Compare the hyperparameters at the top of the next cell to GPT-3's, for a sense of scale: `n_embd`=64 here vs. 12288, `n_layer`=4 vs. 96, `n_head`=4 vs. 96, `block_size`=32 vs. 2048 — same architecture, roughly 800,000× fewer parameters (0.2M vs. ~175B). Everything you need conceptually to build GPT-3 is already in this cell; the rest is scale, systems engineering, and — for a chat-style model — instruction-tuning/RLHF on top.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'   # use a GPU if one is available, else fall back to CPU
eval_iters = 200
n_embd = 64        # width of the token/position embeddings and the residual stream used throughout the model
n_head = 4         # number of parallel attention heads per block; n_embd must be evenly divisible by n_head
n_layer = 4        # number of stacked Transformer blocks
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))       # batch_size random starting offsets
    x = torch.stack([data[i:i+block_size] for i in ix])             # (B, T) input chunks
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])         # (B, T) targets, shifted by 1 position
    x, y = x.to(device), y.to(device)          # move to GPU if available
    return x, y

@torch.no_grad()          # no need to track gradients for an evaluation-only pass -- saves memory and compute
def estimate_loss():
    out = {}
    model.eval()           # switch off dropout for a stable, deterministic-ish eval loss
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()      # average over eval_iters batches -> a much less noisy estimate than a single batch
    model.train()           # switch back to training mode (re-enables dropout) before returning
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)      # x -> "what do I offer, to be matched against"
        self.query = nn.Linear(n_embd, head_size, bias=False)    # x -> "what am I looking for"
        self.value = nn.Linear(n_embd, head_size, bias=False)    # x -> "what do I actually send if attended to"
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))    # causal mask, saved with the model but never trained

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C) -> (B,T,head_size)
        q = self.query(x) # (B,T,C) -> (B,T,head_size)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)      -- q_i . k_j, scaled by 1/sqrt(head_size)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)         -- block position i from seeing any j>i
        wei = F.softmax(wei, dim=-1) # (B, T, T)                                          -- each row: a distribution over allowed j's
        wei = self.dropout(wei)                                                          # randomly zero some attention weights while training (regularization)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C) -> (B,T,head_size)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)          -- weighted sum of value vectors, weights = attention
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])   # num_heads independent heads, all run on the same x
        self.proj = nn.Linear(n_embd, n_embd)     # combine the concatenated heads back into a single n_embd-dim vector
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # each head outputs (B,T,head_size); concat along last dim -> (B,T,n_embd)
        out = self.dropout(self.proj(out))                     # linear projection back to n_embd, then dropout
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand: n_embd -> 4*n_embd (extra capacity to compute in)
            nn.ReLU(),                        # non-linearity -- without it, stacking linear layers would collapse into one linear layer
            nn.Linear(4 * n_embd, n_embd),   # project back down: 4*n_embd -> n_embd (must match the residual stream's width)
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)    # applied identically and independently to every token position -- no cross-token mixing happens here

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head       # split the embedding evenly across heads
        self.sa = MultiHeadAttention(n_head, head_size)    # "communication": tokens exchange information with each other
        self.ffwd = FeedFoward(n_embd)                      # "computation": each token processes what it just received, alone
        self.ln1 = nn.LayerNorm(n_embd)     # normalizes the input going into self-attention
        self.ln2 = nn.LayerNorm(n_embd)     # normalizes the input going into the feed-forward

    def forward(self, x):
        x = x + self.sa(self.ln1(x))       # residual connection: x + (update computed from a normalized copy of x)
        x = x + self.ffwd(self.ln2(x))     # same residual pattern for the feed-forward sublayer
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)         # token id -> n_embd-dim vector ("what token is this")
        self.position_embedding_table = nn.Embedding(block_size, n_embd)     # position index -> n_embd-dim vector ("where is it")
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])   # stack of n_layer Transformer blocks
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)      # project the final n_embd-dim vector down to vocab_size logits

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)         -- what token is at each position
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)     -- where each position is
        x = tok_emb + pos_emb # (B,T,C)             -- broadcast-add position info onto every sequence in the batch
        x = self.blocks(x) # (B,T,C)                -- run through all n_layer Transformer blocks
        x = self.ln_f(x) # (B,T,C)                  -- final normalization before reading out logits
        logits = self.lm_head(x) # (B,T,vocab_size)  -- unnormalized scores for "what comes next" at every position

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)          # flatten (B,T) into one axis so cross_entropy sees (N, vocab_size)
            targets = targets.view(B*T)           # flatten targets to match, (N,)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]        # the model only ever learned positions 0..block_size-1, so never feed it more context than that
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)          -- only the very next token's prediction is needed
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)      -- stochastic sampling, not argmax, so output varies run to run
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)     # clear stale gradients left from the previous step
    loss.backward()                            # backprop: compute d(loss)/d(param) for every parameter
    optimizer.step()                           # AdamW update using those gradients

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)     # start generation from a single "beginning" token (id 0, i.e. '\n')
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4198, val loss 2.4340
step 400: train loss 2.3503, val loss 2.3567
step 500: train loss 2.2970, val loss 2.3136
step 600: train loss 2.2410, val loss 2.2506
step 700: train loss 2.2062, val loss 2.2198
step 800: train loss 2.1638, val loss 2.1871
step 900: train loss 2.1232, val loss 2.1494
step 1000: train loss 2.1020, val loss 2.1293
step 1100: train loss 2.0704, val loss 2.1196
step 1200: train loss 2.0382, val loss 2.0798
step 1300: train loss 2.0249, val loss 2.0640
step 1400: train loss 1.9922, val loss 2.0354
step 1500: train loss 1.9707, val loss 2.0308
step 1600: train loss 1.9614, val loss 2.0474
step 1700: train loss 1.9393, val loss 2.0130
step 1800: train loss 1.9070, val loss 1.9943
step 1900: train loss 1.9057, val loss 1.9871
step 2000: train loss 1.8834, val loss 1.9954
step 2100: train loss 1.

### Tracing one token through the full model

Concrete shapes make this much less abstract. Take a single training example: `T = 32` tokens (`block_size`); batch size is irrelevant here, so drop `B` and just track one sequence.

1. **Input**: a sequence of 32 integer token ids, each in `[0, 65)`.
2. **Embedding**: `token_embedding_table` maps each id to a 64-dim vector → shape `(32, 64)`. `position_embedding_table` adds a learned 64-dim vector *per position* (0..31) → still `(32, 64)`, now position-aware.
3. **Inside one `Block`**: `LayerNorm` normalizes each of the 32 rows independently → `(32, 64)` unchanged in shape. `MultiHeadAttention` splits the 64 channels across `n_head=4` heads of `head_size=16` each: each head computes `(32,32)` attention weights and outputs `(32,16)`; the 4 heads' outputs are concatenated back to `(32,64)`, then linearly projected `(32,64) → (32,64)`. Added back onto the block's input (residual). Then `LayerNorm` again, then `FeedFoward` expands to `(32,256)` and back down to `(32,64)`, again added back via residual.
4. **Repeat step 3** `n_layer=4` times — same shape in, same shape out, at every block; only the *content* of the 64-dim vectors changes, accumulating more context each pass.
5. **Final LayerNorm**: `(32,64) → (32,64)`.
6. **`lm_head`**: one more linear layer, `(32,64) → (32,65)` — 65 logits (one per possible next character) at every one of the 32 positions.
7. **At generation time**, only the logits at the *last* position (`[-1,:]`, shape `(65,)`) get used — turned into probabilities via `softmax` and sampled from via `multinomial` to pick the next token.

Notice the embedding dimension (64) never changes size anywhere in the middle of the network — every block takes `(T, 64)` in and returns `(T, 64)` out. That's a deliberate design choice, and true of every real GPT: a fixed-width "residual stream" that each block reads from and writes back into, only ever changing shape at the very start (embedding) and very end (`lm_head`).

### Where does "0.209729 M parameters" come from?

Walking the parameter count for the hyperparameters used (`vocab_size=65`, `n_embd=64`, `block_size=32`, `n_head=4`, `n_layer=4`):

- **Token embedding**: `vocab_size × n_embd` = 65 × 64 = 4,160
- **Position embedding**: `block_size × n_embd` = 32 × 64 = 2,048
- **Per attention head**: 3 linear projections (`key`, `query`, `value`), each `n_embd × head_size` = 64 × 16 = 1,024, no bias → 3,072 per head. With `n_head=4` heads: 12,288 per block. Plus the output projection `n_embd × n_embd` = 64 × 64 = 4,096 (+64 bias) = 4,160 → **16,448 per block** for attention.
- **Feed-forward**: `n_embd → 4·n_embd` = 64×256 + 256 (bias) = 16,640, then `4·n_embd → n_embd` = 256×64 + 64 = 16,448 → **33,088 per block**.
- **2 LayerNorms per block**: `gamma` + `beta`, each length 64 → 256 per block (small, easy to undercount but negligible here).
- **Per block total**: 16,448 + 33,088 + 256 ≈ 49,792. Over `n_layer=4` blocks: **≈199,168**.
- **Final LayerNorm**: 128.
- **`lm_head`**: `n_embd × vocab_size` + bias = 64×65 + 65 = 4,225.

Total ≈ 4,160 + 2,048 + 199,168 + 128 + 4,225 ≈ **209,729** — matching the printed `0.209729 M parameters` exactly. The two embedding tables and the attention/feed-forward projections inside the 4 blocks dominate; LayerNorm parameters are essentially free. This is the same accounting real GPT parameter counts are built from — plug in GPT-3's hyperparameters (`n_embd`=12288, `n_layer`=96, `vocab_size`≈50k) to see where its ~175B comes from.

### Reading the result

Loss drops from ~4.4 (near random-guess level — recall `-ln(1/65) ≈ 4.17`) to 1.66 train / 1.82 val over 5000 steps, and the generated text — while nonsense as *English* — now has correctly-formed words, plausible character names, stage-direction-like structure, and roughly right punctuation, all learned purely from character sequences, with attention as the only mechanism for using context beyond the immediately preceding character.

**Where to go next, to build this yourself from scratch:**
1. Re-implement each class in this notebook from memory, in a plain `.py` file, without looking back — that's the real test of whether you can build it.
2. Swap the character tokenizer for a real one (`tiktoken`'s GPT-2 BPE) and rerun — `vocab_size` jumps to ~50k, so the embedding/`lm_head` tables get much bigger, but nothing else in the architecture changes.
3. Read Karpathy's [`nanoGPT`](https://github.com/karpathy/nanoGPT) — the same architecture, scaled up with production concerns (mixed precision, gradient accumulation, learning-rate schedules, multi-GPU training).
4. Read the two papers this notebook is a distillation of: *Attention Is All You Need* (Vaswani et al., 2017) for the architecture, and the GPT-2/GPT-3 papers for the decoder-only, pre-norm variant used here.

### Checkpoint: full architecture

1. In `Block.forward`, why is `x = x + self.sa(self.ln1(x))` written as an addition rather than `x = self.sa(self.ln1(x))`?

<details><summary>Show answer</summary>

That's the residual (skip) connection. Without it, gradients have to flow back through every stacked block's transformation, which vanishes or explodes as depth grows. With `x + f(x)`, each block only has to learn a small *update* to `x`, and the original `x` keeps a clean, direct gradient path all the way back to the input — this is what makes deep Transformers trainable at all.

</details>

2. What's the actual difference in what `MultiHeadAttention` computes versus a single `Head` with `head_size = n_embd`?

<details><summary>Show answer</summary>

A single big head produces exactly one attention pattern (one softmax distribution) per query position. `MultiHeadAttention` with `n_head` heads of `head_size = n_embd / n_head` runs `n_head` *independent* Q/K/V projections and *independent* softmaxes in parallel, then concatenates the results — so the model can attend to different things in different ways simultaneously (e.g. one head tracking local syntax, another tracking long-range references), rather than being forced into a single mixing pattern per layer.

</details>

3. Which part of the model is responsible for the network knowing *where* a token is in the sequence — and why is that needed at all, given how attention works?

<details><summary>Show answer</summary>

The `position_embedding_table`. Attention itself is permutation-invariant — it computes affinities from Q·K content, with no built-in sense of order — so without adding a positional signal, shuffling a sequence's tokens around wouldn't change any token's output at all. Summing token embeddings with position embeddings is what gives every position a distinguishable identity before attention ever runs.

</details>

4. If you doubled `n_layer` and `n_embd` but left `block_size` unchanged, what would get more expensive, and what would stay the same cost?

<details><summary>Show answer</summary>

More expensive: everything driven by embedding width — the attention projections and feed-forward layers scale roughly with `n_embd²` per layer, and doubling `n_layer` on top multiplies total compute/parameters further (rough ballpark: 2× layers × ~4× per-layer cost ≈ 8×). Unchanged: anything driven purely by sequence length — the attention matrix stays `(block_size, block_size)` per head, the position embedding table still has only `block_size` rows, and the model still can't see further back than `block_size` tokens of context.

</details>

If you can answer all four without re-reading the code, you understand the architecture well enough to reimplement it.

### FAQ / common pitfalls

- **"My loss isn't decreasing."** Check, in order: is `zero_grad` called every step (stale gradients accumulate silently otherwise)? Is the learning rate absurdly high/low? Is `targets` actually shifted by one position relative to `x` (an off-by-one here silently trains the model to predict the *current* token, which is trivial and won't generalize)?
- **"My attention output looks the same regardless of token order."** You probably forgot the positional embedding — without `position_embedding_table`, attention alone has no notion of order (it only sees which tokens are present and how they relate, not where), so shuffling a sequence's positions wouldn't change per-token outputs at all.
- **"Training works fine on short sequences but breaks/gives garbage past `block_size`."** Expected — the model (and its position embedding table) was never trained on positions beyond `block_size`. `generate()` handles this by cropping (`idx[:, -block_size:]`) before every forward pass; forgetting that crop is a common bug when adapting this code.
- **"Why does validation loss stay close to training loss the whole time, unlike a lot of ML models I've seen?"** At this scale (0.2M params, 1M characters of data) the model is heavily underfitting, not overfitting — there's far more data than capacity to memorize it. Overfitting becomes visible once you scale `n_embd`/`n_layer` up faster than the dataset.
- **"Do I need the causal mask if I'm not generating text autoregressively?"** No — if the whole input is known upfront (classification, a translation encoder, etc.) you want *bidirectional* attention, i.e. skip the `tril` masking entirely, per the encoder/decoder note above.
- **"Why `-inf` and not just a very negative number?"** `-inf` guarantees `softmax` produces *exactly* 0 for masked positions, with no risk of a tiny nonzero leaking through from a "large but finite" negative number, and no risk of numerical issues if the unmasked values happen to be large too.